<a href="https://colab.research.google.com/github/AYUSHDHANGAR/RAG/blob/main/bakery_rag_chatbot_supabase_vectorstoreindex_llamaindex_only_quarry.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
!pip install -q llama-index-llms-google-genai llama-index-embeddings-google-genai

In [5]:
# 1. nest_asyncio install & apply karein
!pip install -q nest_asyncio

import nest_asyncio
nest_asyncio.apply()

In [8]:

from google.colab import userdata
from llama_index.llms.google_genai import GoogleGenAI
from llama_index.embeddings.google_genai import GoogleGenAIEmbedding
from llama_index.core import Settings

# 1. API Key fetch karein
try:
    GOOGLE_API_KEY = userdata.get('google_api_key')
except Exception:
    try:
        GOOGLE_API_KEY = userdata.get('googel_api_key')
    except Exception:
        GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')

In [9]:
# 2. Modern LLM Setup (GoogleGenAI with gemini-3.6-flash)
llm = GoogleGenAI(
    model="gemini-3.6-flash",
    api_key=GOOGLE_API_KEY,
    temperature=0.2
)

In [10]:
# 3. Modern Embedding Setup (GoogleGenAIEmbedding with gemini-embedding-2)
embed_model = GoogleGenAIEmbedding(
    model_name="gemini-embedding-2",
    api_key=GOOGLE_API_KEY
)

In [12]:
# Global settings me set karein
Settings.llm = llm
Settings.embed_model = embed_model

print("⏳ Testing connection with Google Unified GenAI SDK...")
test_resp = llm.complete("Say 'Shri Ram Bakery AI Engine is Live!'")
test_embed = embed_model.get_text_embedding("Shri Ram Bakery Mathura")

print("\n" + "="*50)
print("✅ LLM Connected:", test_resp.text.strip())
print(f"✅ Embedding Connected! Vector Dimensions: {len(test_embed)}")
print("="*50)


⏳ Testing connection with Google Unified GenAI SDK...



✅ LLM Connected: Shri Ram Bakery AI Engine is Live!
✅ Embedding Connected! Vector Dimensions: 3072


In [13]:
!pip install -q pypdf
!pip install -q supabase

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 388.1/388.1 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.4/48.4 kB 4.4 MB/s eta 0:00:00


In [14]:
from supabase import create_client, Client
from llama_index.core import Document

# 1. Exact Working URL
SUPABASE_URL = "https://drrwmheivhcxfvkstcus.supabase.co"

# 2. Apni anon key yahan dalein
SUPABASE_KEY = ""  # <--- Yahan paste karein

supabase: Client = create_client(SUPABASE_URL, SUPABASE_KEY)
print("🔗 Connected to Supabase Project: ")

# 3. Tables Se Data Fetch & LlamaIndex Documents Creation
all_documents = []

tables_to_fetch = [
    ("cakes", "Cakes"),
    ("patties", "Patties"),
    ("commercial_bases", "Commercial Bases"),
    ("celebration_accessories", "Celebration Accessories"),
    ("fast_food_pantry", "Fast Food Pantry"),
    ("confectionery_treats", "Confectionery"),
    ("custom_themes", "Custom Themes"),
    ("delivery_zones", "Delivery Zones"),
    ("store_faqs_and_policies", "Store FAQs"),
    ("products_price_inr", "Pricing Master")
]

print("\n⏳ Fetching live catalog data from Supabase...")

for table_name, category_label in tables_to_fetch:
    try:
        response = supabase.table(table_name).select("*").execute()
        rows = response.data

        for r in rows:
            content_text = r.get('rag_chunk') or r.get('description') or r.get('detailed_policy') or str(r)

            doc = Document(
                text=str(content_text),
                metadata={
                    "sku_id": str(r.get('sku_id') or r.get('id') or r.get('theme_id') or r.get('zone_id') or r.get('faq_id') or ""),
                    "name": str(r.get('name') or r.get('item_name') or r.get('product_name') or r.get('theme_name') or r.get('question') or ""),
                    "category": category_label,
                    "is_eggless": bool(r.get('is_eggless', True)),
                    "thermal_type": str(r.get('thermal_type') or r.get('thermal_classification') or "")
                }
            )
            all_documents.append(doc)

        print(f"✅ Loaded {len(rows):3d} records from '{table_name}'")
    except Exception as e:
        print(f"⚠️ Table '{table_name}' skip hui: {e}")

print("\n" + "="*55)
print(f"🎉 TOTAL LIVE DOCUMENTS INGESTED FROM SUPABASE: {len(all_documents)}")
if len(all_documents) > 0:
    print("\nSample Document 0 Content:\n", all_documents[0].text[:160], "...")
print("="*55)

🔗 Connected to Supabase Project: drrwmheivhcxfvkstcus

⏳ Fetching live catalog data from Supabase...
✅ Loaded 220 records from 'cakes'
✅ Loaded  24 records from 'patties'
✅ Loaded  19 records from 'commercial_bases'
✅ Loaded  16 records from 'celebration_accessories'
✅ Loaded  14 records from 'fast_food_pantry'
✅ Loaded  14 records from 'confectionery_treats'
✅ Loaded   8 records from 'custom_themes'
✅ Loaded   4 records from 'delivery_zones'
✅ Loaded   6 records from 'store_faqs_and_policies'
✅ Loaded 307 records from 'products_price_inr'

🎉 TOTAL LIVE DOCUMENTS INGESTED FROM SUPABASE: 632

Sample Document 0 Content:
 Item: Vanilla Khoaa Cake | Base: Khoaa | Dietary: 100% Pure Vegetarian (Eggless) | Flavor: vanilla | Weight: 0.5kg | Servings: Serves 4-6 people (ideal for smal ...


In [16]:
import pypdf
from llama_index.core import Document
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core.schema import TextNode

pdf_path = "/content/Shri Ram Bakery - Comprehensive Business Specification & Digital Master Blueprint (1).pdf"

# 1. pypdf se clean text extract karein
print("⏳ Extracting clean human text from Master Blueprint PDF...")
reader = pypdf.PdfReader(pdf_path)

pdf_docs = []
for i, page in enumerate(reader.pages):
    page_text = page.extract_text()
    if page_text and page_text.strip():
        doc = Document(
            text=page_text.strip(),
            metadata={"page_num": i + 1, "category": "Master Operational Blueprint"}
        )
        pdf_docs.append(doc)

print(f"✅ Successfully extracted {len(pdf_docs)} clean text pages!")

# 2. 512-Token SentenceSplitter Chunking
splitter = SentenceSplitter(chunk_size=512, chunk_overlap=50)
pdf_nodes = splitter.get_nodes_from_documents(pdf_docs)

for node in pdf_nodes:
    node.metadata["category"] = "Master Operational Blueprint"

# 3. Supabase Tables (632 Records) + Clean PDF Nodes ko merge karein
final_nodes = []

for doc in all_documents:
    node = TextNode(text=doc.text, metadata=doc.metadata)
    final_nodes.append(node)

# Clean PDF nodes add karein
final_nodes.extend(pdf_nodes)

print("\n" + "="*55)
print(f"🎉 FINAL 100% CLEAN KNOWLEDGE BASE READY!")
print(f"   • Supabase Table Records: {len(all_documents)}")
print(f"   • Clean PDF Policy Nodes: {len(pdf_nodes)}")
print(f"   • TOTAL CHUNKS READY FOR INDEXING: {len(final_nodes)}")
print("="*55)

# Verification Preview (Ab asli text aayega!)
print("\n📄 Verified Clean PDF Node 0 Preview:")
print(pdf_nodes[0].text[:160].strip(), "...")

⏳ Extracting clean human text from Master Blueprint PDF...
✅ Successfully extracted 7 clean text pages!

🎉 FINAL 100% CLEAN KNOWLEDGE BASE READY!
   • Supabase Table Records: 632
   • Clean PDF Policy Nodes: 14
   • TOTAL CHUNKS READY FOR INDEXING: 646

📄 Verified Clean PDF Node 0 Preview:
SHRI RAM BAKERY
Master Operational Blueprint, Product Taxonomy & Digital Strategy
Established: Circa 2005 (~21 Years of Heritage)
Compliance: 100% FSSAI Registe ...


In [17]:
!pip install -q rank-bm25 llama-index-retrievers-bm25

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.0/75.0 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 678.4/678.4 kB 25.9 MB/s eta 0:00:00


In [23]:
import time
from llama_index.core.schema import TextNode
from llama_index.core import VectorStoreIndex

print("⏳ Step 1: Preparing 1-by-1 Atomic Nodes (Zero Duplication)...")

# 1. Saari Tables Se Har Ek Row Ko Alag Node Banayein (Pricing Master ko chhod kar)
atomic_nodes = []

for doc in all_documents:
    if doc.metadata.get('category') != 'Pricing Master':
        node = TextNode(
            text=doc.text,
            metadata=doc.metadata
        )
        atomic_nodes.append(node)

# PDF ke 14 policy nodes add karein
atomic_nodes.extend(pdf_nodes)

total_atomic = len(atomic_nodes)
print(f"✅ Total Atomic 1-by-1 Chunks Prepared: {total_atomic}")
print(f"⏱️ Estimated Time: ~3.5 to 4 minutes (Respecting Google's 100 requests/min quota)\n")

# 2. Crash-Proof Rate-Limited Embedding Loop
print("⏳ Step 2: Embedding Chunks 1-by-1 (Ruk-Ruk Kar Safe Speed Par)...")

for i, node in enumerate(atomic_nodes):
    success = False
    retries = 0
    while not success and retries < 10:
        try:
            # 1-by-1 individual embedding
            node.embedding = embed_model.get_text_embedding(node.get_content())
            success = True

            # Har 20 chunks par progress print karein
            if (i + 1) % 20 == 0 or (i + 1) == total_atomic:
                print(f"   progress: [{i+1}/{total_atomic}] chunks safely embedded...")

            time.sleep(0.7)  # 0.7 second gap = ~80 requests/min (100 limit ke andar safe)
        except Exception as e:
            retries += 1
            print(f"   ⚠️ Rate limit spike at chunk {i+1}. Pausing 10 seconds to cool down...")
            time.sleep(10)

print("\n" + "="*55)
print(f"🎉 SAARE {total_atomic} ATOMIC CHUNKS 1-BY-1 EMBED HO GAYE!")
print("="*55)

# 3. Vector Index Create Karein & Disk Par Permanent Save Karein
print("\n⏳ Step 3: Saving Vector Index to './storage_atomic'...")
atomic_vector_index = VectorStoreIndex(atomic_nodes)
atomic_vector_index.storage_context.persist(persist_dir="./storage_atomic")
print("💾 Success! Index permanently saved to './storage_atomic'!")

# 4. Verification Test Query
query_engine = atomic_vector_index.as_query_engine(similarity_top_k=3)
test_q = "Khoaa cake 1kg ka exact rate aur description batao"
print(f"\n🔍 Testing Exact Query: '{test_q}'")
response = query_engine.query(test_q)
print("\n🤖 Bot Response:\n", response)

⏳ Step 1: Preparing 1-by-1 Atomic Nodes (Zero Duplication)...
✅ Total Atomic 1-by-1 Chunks Prepared: 339
⏱️ Estimated Time: ~3.5 to 4 minutes (Respecting Google's 100 requests/min quota)

⏳ Step 2: Embedding Chunks 1-by-1 (Ruk-Ruk Kar Safe Speed Par)...
   progress: [20/339] chunks safely embedded...
   progress: [40/339] chunks safely embedded...
   progress: [60/339] chunks safely embedded...
   progress: [80/339] chunks safely embedded...
   progress: [100/339] chunks safely embedded...
   progress: [120/339] chunks safely embedded...
   progress: [140/339] chunks safely embedded...
   progress: [160/339] chunks safely embedded...
   progress: [180/339] chunks safely embedded...
   progress: [200/339] chunks safely embedded...
   progress: [220/339] chunks safely embedded...
   progress: [240/339] chunks safely embedded...
   progress: [260/339] chunks safely embedded...
   progress: [280/339] chunks safely embedded...
   progress: [300/339] chunks safely embedded...
   progress: [3


🤖 Bot Response:
 1kg Khoaa cake ke rate (keemat) ki jankari uplabdh nahi hai. 

1kg Khoaa cake ke do flavors ki description neeche di gayi hai:

1. **Vanilla Khoaa Cake (1kg):**
   Yeh authentic slow-cooked Indian Khoaa (Mawa / rich condensed milk solids) se taiyar kiya gaya hai, jo ise dense aur melt-in-mouth texture deta hai. Yeh traditional Indian festive sweets aur modern bakery art ka mishran hai. Yeh 100% pure vegetarian aur eggless hai. Ise delicately classic aromatic vanilla bean cream aur sweet gentle notes ke saath infuse kiya gaya hai. (Serves 8-10 people)

2. **Chocolate Khoaa Cake (1kg):**
   Yeh authentic slow-cooked Indian Khoaa (Mawa / rich condensed milk solids) se bana hai, jo ise dense aur melt-in-mouth texture deta hai. Yeh 100% pure vegetarian aur eggless hai. Ise decadent cocoa cream aur rich chocolate fudge ke saath delicately layer kiya gaya hai. (Serves 8-10 people)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')